# Exploring discount patterns

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from Smn.config import PROCESSED_DATA_DIR

clean_df = pd.read_csv(
    PROCESSED_DATA_DIR / "clean_df.csv", parse_dates=["date", "created_date"]
)

### Cost of discounting

In [ ]:
weekly_discount = clean_df.resample("W", on="date").agg(
    total_discount=("discount", "sum"),
    num_orders=("order_id", "nunique"),
)

sns.lineplot(data=weekly_discount.total_discount)
plt.title("Total weekly discount")
plt.show()

In [ ]:
discount_per_order = weekly_discount.total_discount / weekly_discount.num_orders

sns.lineplot(data=discount_per_order)
plt.title("Average discount per order (weekly)")
plt.show()

### Discount impact on sales — top brands

In [ ]:
def top_nine_plots(
    df, top_series, series_col, title,
    plot_x="discount_percentage", plot_y="quantity_sold", freq="d",
):
    """Faceted lmplot showing discount vs quantity for the top 9 in a grouping."""
    plot_data_list = []
    for entry in top_series.index[:9]:
        subset = df[df[series_col] == entry]
        resampled = (
            subset.resample(freq, on="date")
            .agg(
                discount_percentage=("discount_per", "mean"),
                quantity_sold=("product_quantity", "sum"),
                mean_discount=("discount", "mean"),
            )
            .reset_index()
        )
        resampled["log_quantity_sold"] = np.log(resampled.quantity_sold)
        resampled["log_discount"] = np.log(resampled.mean_discount)
        resampled["group"] = entry
        plot_data_list.append(resampled)

    combined = pd.concat(plot_data_list, ignore_index=True)

    g = sns.lmplot(
        data=combined,
        x=plot_x,
        y=plot_y,
        col="group",
        col_wrap=3,
        facet_kws={"sharey": False, "sharex": False},
    )
    g.set_titles("{col_name}")
    g.figure.suptitle(title, fontsize=16, fontweight="bold", y=1.02)

In [ ]:
sorted_brands = clean_df.groupby("long")["revenue"].sum().sort_values(ascending=False)

top_nine_plots(
    clean_df, sorted_brands, "long",
    "Discount Impact on Quantity Sold \u2014 Top 9 Brands",
)

In [ ]:
top_nine_plots(
    clean_df, sorted_brands, "long",
    "Discount vs Sales (log-log, weekly) \u2014 Top 9 Brands",
    "log_discount", "log_quantity_sold", "W",
)

### Discount impact on sales — top products

In [ ]:
top_products = clean_df.groupby("name")["product_quantity"].sum().sort_values(ascending=False)

top_nine_plots(
    clean_df, top_products, "name",
    "Discount vs Sales (log-log, weekly) \u2014 Top 9 Products",
    "log_discount", "log_quantity_sold", "W",
)